<a href="https://colab.research.google.com/github/Rodrigo16Garcia/TFG/blob/main/Voice_Cloning_with_Tacotron_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES
# TO THE CORRECT LOCATION (/kaggle/input) IN YOUR NOTEBOOK,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

import os
import sys
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from urllib.parse import unquote, urlparse
from urllib.error import HTTPError
from zipfile import ZipFile
import tarfile
import shutil

CHUNK_SIZE = 40960
DATA_SOURCE_MAPPING = 'johnoliver:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-data-sets%2F1227149%2F2048443%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20240721%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20240721T185330Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3D887cfd81d42a81b2cf7242580bc5ed997f83c120dc112ce18e3c04735e789ccf5f640654340d8dba9dc4b31ffa68b2fd9ab917b3e4deae63bceb51a7f68a79f30711e72c6fde4e7fc4d88fa8fb7584619256450a14a9261ee4838377aa70013c77b910d7783fa0bb4692f8cb21e03945d76e204277d7e70ef4e5de0c9d69b63000d0849e18f7ce95693677317f2492acc29c958dcba7bf532655c1e3fd86e01ea75218779aba5fbb6d8a22df72b237e9a9f2d6530611319fd34f1e880d749a52c7c46ec480fdd4049703a7523a01b41bdbbb3dd7e1990e23cbb8e715161d3bb80f1a6e648329386c0f0cad4ebf63284d4b1ced54bf648637b288c4abda9aa971'

KAGGLE_INPUT_PATH='/kaggle/input'
KAGGLE_WORKING_PATH='/kaggle/working'
KAGGLE_SYMLINK='kaggle'

!umount /kaggle/input/ 2> /dev/null
shutil.rmtree('/kaggle/input', ignore_errors=True)
os.makedirs(KAGGLE_INPUT_PATH, 0o777, exist_ok=True)
os.makedirs(KAGGLE_WORKING_PATH, 0o777, exist_ok=True)

try:
  os.symlink(KAGGLE_INPUT_PATH, os.path.join("..", 'input'), target_is_directory=True)
except FileExistsError:
  pass
try:
  os.symlink(KAGGLE_WORKING_PATH, os.path.join("..", 'working'), target_is_directory=True)
except FileExistsError:
  pass

for data_source_mapping in DATA_SOURCE_MAPPING.split(','):
    directory, download_url_encoded = data_source_mapping.split(':')
    download_url = unquote(download_url_encoded)
    filename = urlparse(download_url).path
    destination_path = os.path.join(KAGGLE_INPUT_PATH, directory)
    try:
        with urlopen(download_url) as fileres, NamedTemporaryFile() as tfile:
            total_length = fileres.headers['content-length']
            print(f'Downloading {directory}, {total_length} bytes compressed')
            dl = 0
            data = fileres.read(CHUNK_SIZE)
            while len(data) > 0:
                dl += len(data)
                tfile.write(data)
                done = int(50 * dl / int(total_length))
                sys.stdout.write(f"\r[{'=' * done}{' ' * (50-done)}] {dl} bytes downloaded")
                sys.stdout.flush()
                data = fileres.read(CHUNK_SIZE)
            if filename.endswith('.zip'):
              with ZipFile(tfile) as zfile:
                zfile.extractall(destination_path)
            else:
              with tarfile.open(tfile.name) as tarfile:
                tarfile.extractall(destination_path)
            print(f'\nDownloaded and uncompressed: {directory}')
    except HTTPError as e:
        print(f'Failed to load (likely expired) {download_url} to path {destination_path}')
        continue
    except OSError as e:
        print(f'Failed to load {download_url} to path {destination_path}')
        continue

print('Data source import complete.')


[=                                                 ] 47800320 bytes downloaded

Clone the repo from GitHub

In [ ]:
!git clone https://github.com/Ryan-Rudes/tacotron2
%cd tacotron2
!git submodule init; git submodule update

Create file lists from train and validation sets

In [ ]:
import pandas as pd

metadata = pd.read_csv("../../input/johnoliver/metadata.csv")
metadata = metadata[metadata['include']]

total = len(metadata)
split = 0.9
train = int(total * split)

train_metadata = metadata[:train]
val_metadata = metadata[train:]

with open('filelists/audio_text_train_filelist.txt', 'w') as f:
    for _, (index, _, _, _, text, _) in train_metadata.iterrows():
        filepath = '../../input/johnoliver/wav/%05d.wav' % index
        f.write(filepath + '|' + text + '\n')

with open('filelists/audio_text_test_filelist.txt', 'w') as f:
    for _, (index, _, _, _, text, _) in val_metadata.iterrows():
        filepath = '../../input/johnoliver/wav/%05d.wav' % index
        f.write(filepath + '|' + text + '\n')

Install dependencies

In [ ]:
!pip install tensorflow==1.15
!pip install unidecode
!pip install inflect

Download pretrained model from Google Drive for transfer learning

In [ ]:
!pip install gdown
!gdown https://drive.google.com/uc?id=1c5ZTuT7J08wLUoVZ2KkUs_VdZuJ86ZqA

Train the model for 1000 epochs with a batch size of 32

In [ ]:
!python train.py --output_directory=outdir --log_directory=logdir -c tacotron2_statedict.pt --warm_start

## Inference
### Synthesize generated audio samples from text

In [ ]:
from multiprocessing import Pool
import matplotlib.pylab as plt
%matplotlib inline
from tqdm.notebook import tqdm
import IPython.display as ipd
from time import time, sleep
import scipy.io.wavfile
import numpy as np
import matplotlib
import torch
import sys

sys.path.append('waveglow/')

from audio_processing import griffin_lim
from layers import TacotronSTFT, STFT
from hparams import create_hparams
from text import text_to_sequence
from denoiser import Denoiser
from train import load_model
from model import Tacotron2

In [ ]:
def plot_data(data, figsize=(16, 4)):
    fig, axes = plt.subplots(1, len(data), figsize=figsize)
    for i in range(len(data)):
        axes[i].imshow(data[i], aspect='auto', origin='lower', interpolation='none')

In [ ]:
hparams = create_hparams()
hparams.sampling_rate = 22000

In [ ]:
checkpoint = int(input("Enter steps at latest checkpoint: "))
checkpoint_path = f"outdir/checkpoint_{checkpoint}"
model = load_model(hparams)
model.load_state_dict(torch.load(checkpoint_path)['state_dict'])
_ = model.cuda().eval().half()

In [ ]:
!gdown https://drive.google.com/uc?id=1rpK8CzAAirq9sWZhe9nlfvxMF1dRgFbF

In [ ]:
waveglow_path = 'waveglow_256channels_universal_v5.pt'
waveglow = torch.load(waveglow_path)['model']
waveglow.cuda().eval().half()
for k in waveglow.convinv:
    k.float()
denoiser = Denoiser(waveglow)

In [ ]:
def synthesize(word, n=5, cleaners=['english_cleaners'], sigma=0.666, strength=0.01):
    sequence = np.array(text_to_sequence(word, cleaners))[None, :]
    sequence = torch.autograd.Variable(torch.from_numpy(sequence)).cuda().long()
    with torch.no_grad():
        mel_outputs, mel_outputs_postnet, _, alignments = model.inference(sequence)
    audio = waveglow.infer(mel_outputs_postnet, sigma=sigma)
    audio_denoised = denoiser(audio, strength=strength)[:, 0]
    audio = audio[0].data.cpu().numpy().tolist()
    mel_outputs = mel_outputs.float().data.cpu().numpy()[0]
    mel_outputs_postnet = mel_outputs_postnet.float().data.cpu().numpy()[0]
    alignments = alignments.float().data.cpu().numpy()[0]
    return audio, mel_outputs, mel_outputs_postnet, alignments

In [ ]:
tests = [
         "Scientists at the CERN laboratory say they have discovered a new particle.",
         "The state of Florida reports a surge in coronavirus deaths as restrictions are upended.",
         "How much wood would a woodchuck chuck if a woodchuck could chuck wood?",
         "A woodchuck would chuck all the wood it could chuck if a woodchuck could chuck wood.",
         "Peter Piper picked a peck of pickled peppers. How many pickled peppers did Peter Piper pick?",
         "Sally sells seashells by the seashore. The shells she sells are seashells I'm sure.",
         "The blue lagoon is a nineteen eighty American romance adventure film.",
         "George Washington was the first President of the United States.",
         "Basilar membrane and otolaryngology are not auto-correlations.",
         "Biden holds first foreign meeting with Canada's Justin Trudeau."
]

In [ ]:
for text in tests:
    audio, mel_outputs, mel_outputs_postnet, alignments = synthesize(text, n=15)
    ipd.display_html(ipd.HTML(f"""
    <h3>{text}</h3>
    <br/>
    """))
    ipd.display(ipd.Audio(data = audio, rate = 22000, autoplay = False))